In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['Canasta_db'] 
coleccion = db['Retail_A'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [10]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
import pandas as pd
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Limpieza de procesos según protocolo de laboratorio
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "LosAveMayo"
datos_finales = []
driver = None

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR (VISIBILIDAD TOTAL) ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")

# IMPORTANTE: Eliminamos --headless para que veas la ventana en el VNC (localhost:6080)
# options.add_argument("--headless") 

options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

try:
    driver = webdriver.Chrome(options=options)
    print("🚀 Navegador iniciado. ¡VE AL VNC AHORA (localhost:6080)!")

    # --- PASO 2: NAVEGACIÓN E INTERVENCIÓN MANUAL ---
    driver.get("https://www.unimarc.cl/category/despensa")
    
    # PROTOCOLO DE RESCATE: Pausa para manejar el pop-up de Unimarc
    print("\n⚠️ ESPERANDO INTERVENCIÓN EN VNC:")
    print("1. En el VNC, selecciona una comuna si aparece el aviso.")
    print("2. Cierra cualquier banner que tape los productos.")
    input("👉 Una vez que veas los productos cargados, presiona ENTER aquí...")

    limite_paginas = 3
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")
        time.sleep(8) 

        WebDriverWait(driver, 25).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "section.smu-impressed"))
        )

        # Scroll progresivo para gatillar la carga de datos
        driver.execute_script("window.scrollTo(0, 1200);")
        time.sleep(3)
        driver.execute_script("window.scrollTo(0, 2500);")
        time.sleep(2)

        bloques = driver.find_elements(By.CSS_SELECTOR, "section.smu-impressed")

        for bloque in bloques:
            try:
                # Selectores basados en el HTML proporcionado
                marca = bloque.find_element(By.CSS_SELECTOR, "p.Shelf_brandText__vmuWJ").text
                producto = bloque.find_element(By.CSS_SELECTOR, "p.Shelf_nameProduct__0KIRG").text
                gramaje = bloque.find_element(By.CSS_SELECTOR, "div#shelf__ppum p").text
                precio_texto = bloque.find_element(By.CSS_SELECTOR, "p[id^='listPrice__offerPrice--']").text

                datos_finales.append({
                    "marca": marca.strip().upper(),
                    "producto": producto.strip(),
                    "gramaje": gramaje.strip(),
                    "valor_sucio": precio_texto,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                continue

        # Paginación
        if nivel_pagina < limite_paginas - 1:
            try:
                pagina_objetivo = nivel_pagina + 2
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(4)
                btn = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, f"//button[text()='{pagina_objetivo}']"))
                )
                driver.execute_script("arguments[0].click();", btn)
                print(f"✅ Cambio a página {pagina_objetivo}")
            except:
                print(f"⚠️ No se pudo pasar a la página {nivel_pagina + 2}")
                break

    print(f" Extracción terminada: {len(datos_finales)} registros.")

except Exception as e:
    print(f" Error en Selenium: {e}")
finally:
    if driver:
        driver.quit()

# --- PASO 3: LIMPIEZA Y GUARDADO EN MONGODB ---
try:
    # Usamos host 'mongodb' para asegurar la conexión en tu contenedor
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["Canasta_db"]
    coleccion = db["Retail_A"]

    if datos_finales:
        for d in datos_finales:
            # Regex para extraer solo los números y evitar ceros
            solo_numeros = re.sub(r'[^\d]', '', d["valor_sucio"])
            d["valor_clp"] = float(solo_numeros) if solo_numeros else 0.0
            del d["valor_sucio"]

        coleccion.insert_many(datos_finales)
        print("✅ Datos guardados en MongoDB.")
    else:
        print(" No hay datos para guardar.")
except Exception as e:
    print(f"❌ Error en MongoDB: {e}")

# --- PASO 4: TABLA FINAL ---
if datos_finales:
    df = pd.DataFrame(datos_finales)
    print("\n" + "="*85)
    print(f"📊 REPORTE DETALLADO - GRUPO {NOMBRE_GRUPO}")
    print("="*85)
    # Mostramos las columnas solicitadas
    columnas = ['producto', 'marca', 'valor_clp', 'gramaje']
    print(df[columnas].to_string(index=False))
    
    promedio = df[df['valor_clp'] > 0]['valor_clp'].mean()
    print("-" * 85)
    print(f"TOTAL ÍTEMS: {len(df)} | PRECIO PROMEDIO: ${promedio:,.0f} CLP")
    print("="*85)

🧹 Limpieza de procesos y temporales completada.
🚀 Navegador iniciado. ¡VE AL VNC AHORA (localhost:6080)!

⚠️ ESPERANDO INTERVENCIÓN EN VNC:
1. En el VNC, selecciona una comuna si aparece el aviso.
2. Cierra cualquier banner que tape los productos.


👉 Una vez que veas los productos cargados, presiona ENTER aquí... 


--- Procesando Página 1 ---
⚠️ No se pudo pasar a la página 2
 Extracción terminada: 40 registros.
✅ Datos guardados en MongoDB.

📊 REPORTE DETALLADO - GRUPO LosAveMayo
                                                          producto           marca  valor_clp gramaje
                        Arroz nuestra cocina g2 largo delgado 1 kg  NUESTRA COCINA     1000.0    1 Kg
                       Aceite nuestra cocina 100% maravilla 900 ml  NUESTRA COCINA     1990.0  900 ml
                      Aceite nuestra cocina vegetal botella 900 ml  NUESTRA COCINA     1790.0  900 ml
     Atún lomito nuestra cocina en agua 160 g neto - 104 g drenado  NUESTRA COCINA     1350.0   104 g
                                      Aceite merkat vegetal 900 ml          MERKAT     1550.0  900 ml
                                      Azúcar granulada iansa 900 g           IANSA     1000.0   900 g
                                Pasta spaghetti n° 5 carozzi 400 g         CAROZZI    21650.0   400 g
               